# PKG Geographic Analytics — EDA & Use-Case Notebook
### Payment Knowledge Graph · Treasury Management · Data Science
**Scope: `version = 'P99_9'` · table `bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics`**

---

## What this notebook is for

Block F emits 24 geographic columns per node per month. This notebook answers
three questions in order, and refuses to skip to the third:

1. **Is the geo block trustworthy?** Coverage gates, guard-NaN rates, and the
   population composition recomputed under `party_type`.
2. **Does geography carry information that the other metrics don't already
   have?** Most candidate "geographic findings" are size restated. The peer
   normalisation layer (§4) exists to strip that out; the correlation
   structure (§5) shows what survives.
3. **What is worth building an application around?** Four candidate use cases
   (§7–§10), each with a persistence requirement and a rung-agreement check.

## Standing constraints this notebook enforces

| Constraint | Where enforced |
|---|---|
| `scope = on_us_c2c` — net flow means *vs other PNC customers*, not vs the economy | stated on every aggregate |
| Never aggregate across `node_type` without declaring it | every groupby carries it |
| Nothing reportable on a single month | §6 persistence, §7–9 require *k* consecutive months |
| Nothing reportable at one rung — check two | §11 P99_9 vs P99 |
| Geo metrics are computed on **located counterparties only** | §2 coverage gate, applied everywhere after |
| Raw dispersion is confounded by degree and industry | §4 peer percentiles; raw values are never the reportable quantity |
| `strength = in + out` double-counts — a weight, not a volume figure | never summed across nodes |

> **Population warning.** At P99_9 the graph is overwhelmingly individual
> nodes. Every business-facing result below filters
> `entity_type = 'business'`. §1 recomputes the exact split under
> `party_type`, which is the first thing to read.

---
## 0. Setup

In [ ]:
import os, math, warnings, textwrap
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Plotly renderer. 'notebook' keeps the figures inside the .ipynb; if the
# figures do not appear on JupyterHub, switch to 'iframe'.
import plotly.io as pio
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Safe

# ---------------------------------------------------------------- config --
# Two ways in. The parquet path is what the pipeline writes; the Hive table is
# the same data registered. Parquet is usually faster (no metastore round
# trip) and works before the table is refreshed.
SOURCE       = "table"         # "table" | "parquet"
HIVE_TABLE   = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
PARQUET_GLOB = "/user/pk36814/metrics/node/*.parquet"

VERSION     = "P99_9"          # primary production rung
VERSION_ALT = "P99"            # adjacent rung, for the §11 agreement check
REF_MONTH   = None             # None -> latest month found in §1
BIZ         = "business"       # entity_type value for organisations

# ---- analysis gates (justified where used; change here, not inline) -----
MIN_GEO_COV   = 0.60    # min located-dollar share for any geo result
MIN_CP        = 5       # min counterparties (matches the entropy guard)

# ---- §7 detector -------------------------------------------------------
# The v1 detector used a peer-percentile cutoff and measured 70.6% false
# discovery against its own placebo. That is not a threshold that was set
# too loose: a percentile cutoff flags a FIXED FRACTION of the population
# whether or not anything happened, so it cannot control false discovery at
# any setting. v2 replaces it with Mann-Kendall (analytic null) plus
# Benjamini-Hochberg. FDR is then controlled by construction and FDR_Q is
# the thing you actually choose.
FDR_Q               = 0.10     # target false-discovery rate
MIN_EFFECT_RATIO    = 1.50     # trailing/leading spread ratio to call it material
MIN_SPREAD_BASELINE = 25.0     # km; below this it is market ENTRY, not widening
MIN_STRENGTH_PANEL  = 100_000  # monthly in+out; excludes the micro tail
MIN_CP_SUSTAINED    = 5        # median located counterparties across months
MIN_MONTHS          = 12       # minimum months present to test a node

# Full population. Set to an int only to prototype; results below assume None.
SAMPLE_LIMIT  = None
PANEL_TOP_N   = None           # None -> every business node in the panel

OUT = "../metrics/eda_geo"
os.makedirs(OUT, exist_ok=True)
print(f"source={SOURCE} | version={VERSION} (alt {VERSION_ALT})")
print(f"full population: SAMPLE_LIMIT={SAMPLE_LIMIT}  PANEL_TOP_N={PANEL_TOP_N}")
print(f"detector: FDR_Q={FDR_Q}  effect>={MIN_EFFECT_RATIO}x  "
      f"baseline>={MIN_SPREAD_BASELINE}km  strength>={MIN_STRENGTH_PANEL:,}")
RESULTS = {}   # every headline number lands here; written out in §12

In [ ]:
# ------------------------------------------------- Spark data access -----
# 96.5M rows across all rungs. Everything heavy stays in Spark; only
# aggregated or explicitly-filtered results are pulled into pandas.
import time
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("pkg_geo_eda")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.execution.arrow.maxRecordsPerBatch", "50000")
         .config("spark.sql.shuffle.partitions", "400")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

SDF = (spark.table(HIVE_TABLE) if SOURCE == "table"
       else spark.read.parquet(PARQUET_GLOB))

# One temp view for the whole notebook. Every SQL cell below reads FROM {TABLE},
# so pointing TABLE at the view means the same SQL runs against either source.
SDF.createOrReplaceTempView("pkg_metrics")
TABLE = "pkg_metrics"

n_all = SDF.count()
print(f"{SOURCE}: {n_all:,} rows x {len(SDF.columns)} columns")

def q(sql, label="", max_rows=5_000_000):
    """Run Spark SQL and return pandas.

    Guarded on purpose: a stray SELECT * against 96.5M rows would try to
    collect the whole table to the driver. Aggregate in Spark, land small
    frames in pandas — that is the whole discipline of this notebook.
    """
    t = time.time()
    sdf = spark.sql(sql)
    n = sdf.count()
    if n > max_rows:
        raise MemoryError(
            f"[{label}] {n:,} rows exceeds max_rows={max_rows:,}. Aggregate "
            f"in Spark first, or raise max_rows deliberately if the driver "
            f"can hold it.")
    df = sdf.toPandas()
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"[{label or 'query'}] {len(df):,} rows x {df.shape[1]} cols "
          f"| {time.time()-t:,.1f}s | {mb:,.1f} MB")
    return df

# Never downcast a coordinate. float16 carries ~3 significant digits, so
# lat 40.44052 becomes 40.4375 — a silent 0.3 km displacement on EVERY node,
# which would corrupt every distance, centroid and drift figure downstream.
NEVER_SHRINK = {"lat", "lon", "geo_centroid_lat", "geo_centroid_lon"}

def shrink(df):
    """Downcast a collected frame. toPandas returns float64/object for
    everything; on a multi-million-row panel that is most of the memory.

    float32 is the floor, never float16: numpy's linalg rejects float16
    outright (so §4 would crash), and where it does not crash it quietly
    destroys precision.
    """
    for c in df.columns:
        if c in NEVER_SHRINK:
            df[c] = df[c].astype("float64")
        elif df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
        elif df[c].dtype == "int64":
            df[c] = pd.to_numeric(df[c], downcast="integer")
        elif df[c].dtype == "object" and df[c].nunique(dropna=True) < len(df) / 4:
            df[c] = df[c].astype("category")
    return df

### 0.1 Schema discovery

Column names are resolved from the table itself rather than assumed. Anything
this notebook needs but cannot find is reported here, and the dependent
sections degrade rather than raising `KeyError` twenty cells later.

In [ ]:
DTYPES  = dict(SDF.dtypes)
ALLCOLS = [c.lower() for c in SDF.columns]
print(f"{len(ALLCOLS)} columns\n")
print(pd.Series(DTYPES).value_counts().rename("n_columns").to_string())

def have(*cands, quiet=False):
    """First candidate present in the table, else None."""
    for c in cands:
        if c in ALLCOLS:
            return c
    if not quiet:
        print(f"  [missing] none of {cands}")
    return None

def haveall(prefix):
    return sorted(c for c in ALLCOLS if c.startswith(prefix))

C = {
    "time":      have("time_key", "month", "yyyymm"),
    "version":   have("version", "ladder_version"),
    "node":      have("node", "mdm_id"),
    # typing / identity
    "node_type": have("node_type"),
    "etype":     have("entity_type"),
    "etype_obs": have("entity_type_observed"),
    "etype_inf": have("entity_type_inferred"),
    "esource":   have("entity_type_source"),
    "eclass":    have("entity_class"),
    "naics2":    have("naics2"),
    "naics_desc":have("naics_desc"),
    "cust_name": have("cust_name", "customer_name"),
    "state":     have("state"),
    "zip3":      have("zip3"),
    "lat":       have("lat"), "lon": have("lon"),
    "geo_status":have("geo_status"),
    "attr_prof": have("attr_profile"),
    # flow
    "in_s":      have("in_strength"), "out_s": have("out_strength"),
    "in_d":      have("in_degree"),   "out_d": have("out_degree"),
    "deg":       have("degree"), "net": have("net_flow"),
    # concentration / structure
    "top_share": have("top_share", "top1_share", quiet=True),
    "pagerank":  have("pagerank", "pagerank_logw", quiet=True),
    "clustering":have("clustering_coef", "clustering", quiet=True),
    "recip":     have("reciprocity_node_w", "reciprocity", quiet=True),
    "hub_exp":   have("hub_exposure", quiet=True),
    "months_act":have("months_active", quiet=True),
}
GEO   = haveall("geo_")
SHARE = haveall("share_")
NAICSM= [c for c in ALLCOLS if c.startswith("naics2_") or c.startswith("same_naics2")]

print(f"\ngeo_* columns  ({len(GEO)}): {GEO}")
print(f"\nshare_* columns ({len(SHARE)})")
print(f"naics mix cols : {NAICSM}")
missing = [k for k, v in C.items() if v is None]
print(f"\nunresolved keys: {missing if missing else 'none'}")

In [ ]:
# ------------------------------------------------------------- helpers --
R_EARTH_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised great-circle distance. Used for centroid drift (§6) and
    the registered-vs-flow gap cross-check (§8)."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def sdiv(a, b, fill=np.nan):
    """Safe divide. np.where(cond, a/b, x) still evaluates a/b everywhere
    and emits the warning anyway — this does not."""
    a = np.asarray(a, dtype="float64"); b = np.asarray(b, dtype="float64")
    out = np.full(a.shape, fill, dtype="float64")
    ok = np.isfinite(a) & np.isfinite(b) & (b != 0)
    np.divide(a, b, out=out, where=ok)
    return out

# --------------------------------------------------- trend statistics ----
try:
    from scipy.special import erfc as _erfc
except Exception:                                   # numpy-only fallback
    def _erfc(x):
        x = np.asarray(x, dtype="float64"); z = np.abs(x)
        t = 1.0 / (1.0 + 0.5 * z)
        y = t * np.exp(-z*z - 1.26551223 + t*(1.00002368 + t*(0.37409196
            + t*(0.09678418 + t*(-0.18628806 + t*(0.27886807 + t*(-1.13520398
            + t*(1.48851587 + t*(-0.82215223 + t*0.17087277)))))))))
        return np.where(x >= 0, y, 2.0 - y)

def _tie_term(row):
    v = row[np.isfinite(row)]
    if v.size == 0:
        return 0.0
    _, c = np.unique(v, return_counts=True)
    c = c[c > 1]
    return float((c * (c - 1) * (2 * c + 5)).sum())

def mann_kendall(M, min_obs=MIN_MONTHS):
    """Vectorised Mann-Kendall trend test on a (nodes x months) matrix.

    Rank-based, so a single large remote payment cannot manufacture a trend,
    and — the reason it is here — it has an ANALYTIC null. Ties are corrected
    for, which matters because geo_spread_km is exactly 0 for a co-located
    counterparty cloud and those zeros tie heavily.
    """
    M = np.asarray(M, dtype="float64")
    S = np.zeros(M.shape[0])
    for i in range(M.shape[1] - 1):
        S += np.nansum(np.sign(M[:, i+1:] - M[:, [i]]), axis=1)
    n = np.isfinite(M).sum(axis=1).astype("float64")
    var = (n*(n-1)*(2*n+5) - np.apply_along_axis(_tie_term, 1, M)) / 18.0
    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(S > 0, (S-1)/np.sqrt(var),
            np.where(S < 0, (S+1)/np.sqrt(var), 0.0))
        p = _erfc(np.abs(z) / np.sqrt(2.0))
    bad = (n < min_obs) | ~np.isfinite(var) | (var <= 0)
    return S, np.where(bad, np.nan, z), np.where(bad, np.nan, p), n

def bh_fdr(p, q=None):
    """Benjamini-Hochberg step-up. Returns (mask, critical p).

    This is what makes the flag count defensible: of the nodes flagged, at
    most q are expected to be false, whatever the population size.
    """
    q = FDR_Q if q is None else q
    p = np.asarray(p, dtype="float64")
    idx = np.flatnonzero(np.isfinite(p))
    if idx.size == 0:
        return np.zeros(p.shape, bool), np.nan
    ps = np.sort(p[idx]); m = ps.size
    passed = ps <= q * np.arange(1, m + 1) / m
    if not passed.any():
        return np.zeros(p.shape, bool), np.nan
    crit = float(ps[np.flatnonzero(passed)[-1]])
    out = np.zeros(p.shape, bool); out[idx] = p[idx] <= crit
    return out, crit

def theilsen_slope_matrix(M):
    """Median pairwise slope per row — the EFFECT SIZE that accompanies the
    Mann-Kendall p-value. Significance and magnitude are different questions
    and get different columns."""
    M = np.asarray(M, dtype="float64"); T = M.shape[1]
    i, j = np.triu_indices(T, k=1)
    with np.errstate(invalid="ignore"):
        sl = (M[:, j] - M[:, i]) / (j - i)
    return np.nanmedian(sl, axis=1)

def geo_gate(df, cov_cols=("geo_cov_amt_in", "geo_cov_amt_out"),
             min_cov=None, min_cp=None, biz_only=True, verbose=True):
    """The standing filter for any geographic result.

    Three independent reasons a geo value is not analysable:
      - the node is not a business (the graph is household-dominated),
      - too little of its flow reached a LOCATED counterparty,
      - too few counterparties for the dispersion guards to be meaningful.
    Applied as one function so no section quietly forgets one.
    """
    min_cov = MIN_GEO_COV if min_cov is None else min_cov
    min_cp  = MIN_CP if min_cp is None else min_cp
    m = pd.Series(True, index=df.index)
    steps = [("start", int(m.sum()))]
    if biz_only and C["etype"] in df:
        m &= df[C["etype"]].eq(BIZ); steps.append(("business", int(m.sum())))
    cov = [c for c in cov_cols if c in df]
    if cov:
        m &= df[cov].max(axis=1).ge(min_cov)
        steps.append((f"geo_cov>={min_cov}", int(m.sum())))
    ncp = [c for c in ("geo_n_cp_located_in", "geo_n_cp_located_out") if c in df]
    if ncp:
        m &= df[ncp].sum(axis=1).ge(min_cp)
        steps.append((f"n_cp>={min_cp}", int(m.sum())))
    if verbose:
        print(" -> ".join(f"{k}: {v:,}" for k, v in steps))
    return df.loc[m]

---
## 1. Data contract gates

Nothing below §1 means anything if these fail. Four checks:

1. **Panel completeness** — 23 months present at this rung, no month
   silently short.
2. **Population composition under `party_type`** — the headline
   individual-share figure was measured on the old name typer and can only
   move down. This is where it gets recomputed.
3. **Geographic coverage** — node-level and dollar-weighted.
4. **Guard-NaN rates** — how much of the geo block is actually populated,
   which determines the analysable population for everything after.

In [ ]:
sql = f"""
SELECT {C['time']} AS time_key,
       {C['version']} AS version,
       COUNT(*)                                   AS n_rows,
       COUNT(DISTINCT {C['node']})                AS n_nodes,
       SUM({C['in_s']})                           AS tot_in_strength,
       SUM({C['out_s']})                          AS tot_out_strength
FROM {TABLE}
GROUP BY 1, 2
ORDER BY 2, 1
"""
panel = q(sql, "panel completeness")
piv = panel.pivot(index="time_key", columns="version", values="n_nodes")
print(piv.to_string())
print("\nmonths per version:\n", panel.groupby("version")["time_key"].nunique().to_string())

MONTHS = sorted(panel.loc[panel.version == VERSION, "time_key"].astype(str))
REF_MONTH = REF_MONTH or MONTHS[-1]
print(f"\n{len(MONTHS)} months at {VERSION}: {MONTHS[0]} .. {MONTHS[-1]}   REF_MONTH={REF_MONTH}")

# in_strength and out_strength must agree in a closed on-us system
tot = panel[panel.version == VERSION]
resid = sdiv(tot.tot_in_strength - tot.tot_out_strength, tot.tot_in_strength)
print(f"closure residual (in vs out): max |{np.nanmax(np.abs(resid)):.2e}| "
      f"— should be ~0 for an internally closed graph")

In [ ]:
fig = px.line(panel, x="time_key", y="n_nodes", color="version", markers=True,
              title="Node count by ablation rung — panel completeness check",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=380, xaxis_title=None, yaxis_title="distinct nodes",
                  legend_title="rung")
fig.show()

### 1.2 Population composition under `party_type` — recompute before citing

The previous headline (94.3% individual by node, 48.4% of dollars) was
measured when `entity_type` was inferred from `customer_name`. Under
`party_type` the correction runs **one way only** — person-named
organisations (trusts, estates, single-member LLCs) move individual →
business, never the reverse. This cell produces the number that replaces it.

In [ ]:
sql = f"""
SELECT {C['time']}      AS time_key,
       {C['node_type']} AS node_type,
       COUNT(*)                         AS n_nodes,
       SUM({C['in_s']} + {C['out_s']})  AS strength
FROM {TABLE}
WHERE {C['version']} = '{VERSION}'
GROUP BY 1, 2
"""
comp = q(sql, "composition by node_type")
comp["node_share"]  = comp.groupby("time_key")["n_nodes"].transform(lambda s: s / s.sum())
comp["dollar_share"] = comp.groupby("time_key")["strength"].transform(lambda s: s / s.sum())

latest = comp[comp.time_key.astype(str) == REF_MONTH].sort_values("n_nodes", ascending=False)
print(f"--- {REF_MONTH} @ {VERSION} ---")
print(latest[["node_type", "n_nodes", "node_share", "dollar_share"]].to_string(index=False))
ind = latest.loc[latest.node_type == "individual"]
if len(ind):
    print(f"\nINDIVIDUAL SHARE: {ind.node_share.iloc[0]:.2%} of nodes, "
          f"{ind.dollar_share.iloc[0]:.2%} of dollars")
    print("Compare against the pre-party_type figures (94.26% / 48.4%). "
          "The delta is the person-named-organisation correction.")

In [ ]:
long = comp.melt(id_vars=["time_key", "node_type"],
                 value_vars=["node_share", "dollar_share"],
                 var_name="basis", value_name="share")
fig = px.area(long.sort_values("time_key"), x="time_key", y="share",
              color="node_type", facet_col="basis",
              category_orders={"basis": ["node_share", "dollar_share"]},
              title=f"Population composition over time @ {VERSION} — "
                    f"nodes vs dollars (scope: on_us_c2c)",
              color_discrete_sequence=PALETTE)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=430, xaxis_title=None, legend_title="node_type")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
# How much did party_type actually change? Observed vs inferred, one month.
if C["etype_obs"] and C["etype_inf"]:
    sql = f"""
    SELECT {C['etype_obs']} AS observed, {C['etype_inf']} AS inferred,
           COUNT(*) AS n, SUM({C['in_s']} + {C['out_s']}) AS strength
    FROM {TABLE}
    WHERE {C['version']} = '{VERSION}' AND {C['time']} = '{REF_MONTH}'
    GROUP BY 1, 2
    """
    dis = q(sql, "observed vs inferred")
    dis["pct_nodes"] = dis.n / dis.n.sum()
    dis["pct_dollars"] = dis.strength / dis.strength.sum()
    mat = dis.pivot(index="observed", columns="inferred", values="pct_nodes").fillna(0)
    fig = px.imshow(mat, text_auto=".2%", aspect="auto", color_continuous_scale="Blues",
                    title="Entity type: declared (party_type) vs inferred (name) "
                          f"— share of nodes, {REF_MONTH}")
    fig.update_layout(height=380, xaxis_title="inferred from customer_name",
                      yaxis_title="observed from party_type")
    fig.show()
    key = dis[(dis.observed == "business") & (dis.inferred == "individual")]
    if len(key):
        print(f"PERSON-NAMED ORGANISATIONS: {key.pct_nodes.iloc[0]:.2%} of nodes, "
              f"{key.pct_dollars.iloc[0]:.2%} of dollars.")
        print("These were counted as households under the old typer. They are "
              "trusts, estates, single-member LLCs and sole proprietorships — "
              "and they are a segment in their own right, not just an error bar.")

### 1.3 Geographic coverage and guard-NaN rates

Every Block F metric is computed on **located counterparties only**, and the
dispersion metrics are deliberately NaN below their guards (`geo_spread_*`
below 2 counterparties, entropy below 5). The analysable population is
therefore smaller than the node count, and by how much is not optional
context — it is the denominator for everything in §7–§10.

In [ ]:
gsel = [c for c in ["geo_spread_km", "geo_zip3_entropy", "geo_reach_mean_km",
                    "geo_reach_p90_km", "geo_registered_vs_flow_km",
                    "geo_home_zip3_share_in", "geo_home_state_share_in",
                    "geo_cov_amt_in", "geo_cov_amt_out"] if c in GEO]
nn = ",\n       ".join(
    f"SUM(CASE WHEN {c} IS NOT NULL THEN 1 ELSE 0 END) AS nn_{c}" for c in gsel)
sql = f"""
SELECT {C['node_type']} AS node_type,
       COUNT(*) AS n_nodes,
       {nn},
       AVG({'geo_cov_amt_in' if 'geo_cov_amt_in' in GEO else 'NULL'}) AS avg_cov_in,
       AVG({'geo_cov_amt_out' if 'geo_cov_amt_out' in GEO else 'NULL'}) AS avg_cov_out
FROM {TABLE}
WHERE {C['version']} = '{VERSION}' AND {C['time']} = '{REF_MONTH}'
GROUP BY 1
"""
cov = q(sql, "guard-NaN rates")
for c in gsel:
    cov[f"pop_{c}"] = cov[f"nn_{c}"] / cov.n_nodes
popcols = [f"pop_{c}" for c in gsel]
print(cov[["node_type", "n_nodes", "avg_cov_in", "avg_cov_out"] + popcols].to_string(index=False))

hm = cov.set_index("node_type")[popcols].rename(columns=lambda c: c.replace("pop_geo_", ""))
fig = px.imshow(hm, text_auto=".0%", aspect="auto", color_continuous_scale="Greens",
                title=f"Geo block: share of nodes with a non-null value, by node_type "
                      f"({REF_MONTH} @ {VERSION})")
fig.update_layout(height=340, xaxis_title=None, yaxis_title=None)
fig.show()
print("\nA low bar here is a GUARD, not a data-quality failure: spread is NaN "
      "below 2 counterparties and entropy below 5, deliberately.")

---
## 2. The reference cross-section

One month, all node types, the columns the rest of the notebook needs. Pulled
once and reused. Everything geographic downstream passes through `geo_gate()`.

In [ ]:
base = [C[k] for k in ("node","time","node_type","etype","etype_obs","eclass",
                       "naics2","naics_desc","cust_name","state","zip3","lat","lon",
                       "geo_status","attr_prof","in_s","out_s","in_d","out_d",
                       "net","top_share","pagerank","clustering","recip",
                       "hub_exp","months_act") if C.get(k)]
want = list(dict.fromkeys(base + GEO + SHARE + NAICSM))
sel  = ",\n       ".join(want)
lim  = f"LIMIT {SAMPLE_LIMIT}" if SAMPLE_LIMIT else ""
# Full month, every node. ~1.16M rows at P99_9; shrink() lands it in a few
# hundred MB. No sampling anywhere in this notebook — a sampled denominator
# makes every share and every peer group approximate for no benefit here.

sql = f"""
SELECT {sel}
FROM {TABLE}
WHERE {C['version']} = '{VERSION}'
  AND {C['time']}    = '{REF_MONTH}'
{lim}
"""
X = q(sql, f"cross-section {REF_MONTH}", max_rows=3_000_000)
X.columns = [c.lower() for c in X.columns]
X = shrink(X)
for c in (C["node_type"], C["etype"], C["naics2"], C["state"], "geo_locality_class"):
    if c and c in X: X[c] = X[c].astype("category")
X["strength"] = X[C["in_s"]].fillna(0) + X[C["out_s"]].fillna(0)
X["deg_tot"]  = X[C["in_d"]].fillna(0) + X[C["out_d"]].fillna(0)
print(X.shape)
X.head(3)

In [ ]:
# The analysable business population, and what each gate costs.
G = geo_gate(X)
print(f"\nanalysable business population: {len(G):,} of {len(X):,} sampled rows")
print(f"share of business dollars retained: "
      f"{G.strength.sum() / max(X.loc[X[C['etype']].eq(BIZ), 'strength'].sum(), 1):.1%}")

### 2.1 Is coverage confounding dispersion?

`geo_spread_km` is computed over located counterparties. If located
counterparties are systematically nearer or farther than unlocated ones,
spread and coverage will be correlated and the `MIN_GEO_COV` gate is doing
real work rather than being decorative. Worth knowing which.

In [ ]:
if "geo_cov_amt_in" in X and "geo_spread_km" in X:
    d = X[X[C["etype"]].eq(BIZ)].dropna(subset=["geo_spread_km", "geo_cov_amt_in"]).copy()
    d["cov_bin"] = pd.cut(d.geo_cov_amt_in, np.arange(0, 1.05, 0.1))
    agg = (d.groupby("cov_bin", observed=True)
             .agg(n=("geo_spread_km", "size"),
                  median_spread=("geo_spread_km", "median"),
                  p90_spread=("geo_spread_km", lambda s: s.quantile(0.90)))
             .reset_index())
    agg["cov_bin"] = agg.cov_bin.astype(str)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=agg.cov_bin, y=agg.n, name="nodes", marker_color="#cfd8dc")
    fig.add_scatter(x=agg.cov_bin, y=agg.median_spread, name="median spread km",
                    mode="lines+markers", line=dict(width=3), secondary_y=True)
    fig.add_scatter(x=agg.cov_bin, y=agg.p90_spread, name="p90 spread km",
                    mode="lines+markers", line=dict(dash="dot"), secondary_y=True)
    fig.add_vline(x=int(MIN_GEO_COV * 10) - 0.5, line_dash="dash", line_color="crimson")
    fig.update_layout(height=400, title="Does located-dollar coverage bias measured "
                      "dispersion? (business nodes)", xaxis_title="geo_cov_amt_in bin")
    fig.update_yaxes(title_text="nodes", secondary_y=False)
    fig.update_yaxes(title_text="km", secondary_y=True)
    fig.show()
    lo = d[d.geo_cov_amt_in < MIN_GEO_COV].geo_spread_km.median()
    hi = d[d.geo_cov_amt_in >= MIN_GEO_COV].geo_spread_km.median()
    print(f"median spread below the gate: {lo:,.0f} km | at or above: {hi:,.0f} km "
          f"| ratio {sdiv(lo, hi):.2f}")
    print("A ratio far from 1.0 means low-coverage nodes are not a random "
          "subsample and the gate is load-bearing. Near 1.0 means the gate is "
          "cheap insurance and can be relaxed if it costs too much population.")

---
## 3. What the geographic block looks like

Distributions first, by `node_type`, because the households and the businesses
are different populations and a pooled histogram is a statement about
households.

In [ ]:
dist_cols = [c for c in ["geo_spread_km", "geo_reach_p50_km", "geo_reach_p90_km",
                         "geo_zip3_entropy", "geo_n_zip3_80",
                         "geo_registered_vs_flow_km"] if c in X]
d = X[X[C["node_type"]].isin(["individual", "business_naics_valid",
                              "business_naics_missing"])]
fig = make_subplots(rows=2, cols=3, subplot_titles=dist_cols)
for i, c in enumerate(dist_cols):
    r, k = divmod(i, 3)
    for j, nt in enumerate(d[C["node_type"]].cat.remove_unused_categories().cat.categories):
        v = d.loc[d[C["node_type"]].eq(nt), c].dropna()
        if len(v) < 100: continue
        v = np.log10(v.clip(lower=0.1)) if c.endswith("_km") else v
        fig.add_histogram(x=v, name=str(nt), legendgroup=str(nt),
                          showlegend=(i == 0), opacity=0.55, nbinsx=60,
                          marker_color=PALETTE[j], row=r + 1, col=k + 1)
fig.update_layout(height=620, barmode="overlay",
                  title="Geo block distributions by node_type "
                        f"({REF_MONTH} @ {VERSION}) — km axes are log10")
fig.show()
print("Read the km panels as log10: 2.0 = 100 km, 3.0 = 1,000 km.")

In [ ]:
if "geo_locality_class" in X and C["naics2"]:
    b = X[X[C["etype"]].eq(BIZ)].dropna(subset=["geo_locality_class"])
    top = b[C["naics2"]].value_counts().head(14).index
    ct = (pd.crosstab(b.loc[b[C['naics2']].isin(top), C["naics2"]],
                      b.loc[b[C['naics2']].isin(top), "geo_locality_class"],
                      normalize="index")
          .reindex(columns=["LOCAL", "REGIONAL", "MULTI_MARKET", "NATIONAL"]))
    fig = px.imshow(ct, text_auto=".0%", aspect="auto", color_continuous_scale="Purples",
                    title="Locality class by NAICS2 sector — business nodes. "
                          "The row profile IS the industry footprint signature")
    fig.update_layout(height=520, xaxis_title=None, yaxis_title="naics2")
    fig.show()
    print("Rows that look alike are sectors with the same footprint shape. A "
          "customer whose class disagrees with its sector row is the anomaly "
          "candidate — that is the seed of the peer percentile in §4.")

---
## 4. Peer normalisation — the missing layer

The manifest is explicit that **raw dispersion is not the reportable
quantity**: a node with four counterparties *cannot* have high ZIP3 entropy,
and a large firm mechanically reaches farther. Block F emits raw values and
leaves normalisation as a downstream step. This section is that step.

**Method.** For each geo metric:

1. Regress `log1p(metric)` on `log1p(degree)` and `log1p(strength)` — OLS,
   fitted **within `node_type`** so a household's size-distance relationship
   is not imposed on a business.
2. Take the residual — the part of the footprint not explained by size.
3. Percentile-rank the residual within **`naics2` × size-decile ×
   `node_type`**, requiring a minimum peer-group size.

The output `{metric}_pctile_naics_size` is what any downstream alert,
scorecard or app should consume. Raw km values are for display only.

In [ ]:
from numpy.linalg import lstsq

PEER_MIN = 30      # smallest peer group we will percentile within

def peer_percentile(df, metric, size_cols=None, peer_cols=None,
                    peer_min=PEER_MIN, logy=True):
    """Residualise `metric` on size, then percentile-rank within peers.

    Returns (residual, percentile). Percentile is NaN where the peer group is
    too small to rank against — an unrankable node is not a median node, and
    filling it with 0.5 would invent a peer comparison that was never made.
    """
    size_cols = size_cols or ["deg_tot", "strength"]
    peer_cols = peer_cols or [C["naics2"], "size_decile", C["node_type"]]
    d = df[[metric] + size_cols + [c for c in peer_cols if c in df]].copy()
    y_ok = d[metric].notna() & np.isfinite(d[metric])
    y = (np.log1p(d.loc[y_ok, metric].clip(lower=0)) if logy
         else d.loc[y_ok, metric]).astype("float64")
    # float64 explicitly: lstsq refuses float16 and loses conditioning on
    # float32, and a downcast frame can arrive here as either.
    Xd = np.column_stack([np.ones(y_ok.sum())] +
                         [np.log1p(d.loc[y_ok, c].fillna(0).clip(lower=0)
                                   ).astype("float64") for c in size_cols])
    beta, *_ = lstsq(Xd, y.to_numpy(), rcond=None)
    resid = pd.Series(np.nan, index=d.index)
    resid.loc[y_ok] = y.to_numpy() - Xd @ beta
    r2 = 1 - np.nanvar(resid.loc[y_ok]) / max(np.nanvar(y), 1e-12)
    grp = d.assign(_r=resid).groupby([c for c in peer_cols if c in d], observed=True)["_r"]
    pct = grp.rank(pct=True)
    pct[grp.transform("size") < peer_min] = np.nan
    return resid, pct, r2, beta

B = G.copy()
B["size_decile"] = pd.qcut(B.strength.rank(method="first"), 10,
                           labels=False, duplicates="drop")
# geo_n_zip3_80 is a COUNT bounded by the counterparty count, so OLS on
# log-degree overshoots: v1 drove its degree correlation from +0.138 to
# -0.249, i.e. it manufactured the opposite confound. A count bounded by a
# denominator is normalised as a RATIO to that denominator, not residualised
# against it.
if "geo_n_zip3_80" in B:
    B["geo_zip3_per_cp"] = sdiv(B["geo_n_zip3_80"], B["deg_tot"].clip(lower=1))

# geo_zip3_entropy and geo_n_zip3_80 correlate at 0.85 after normalisation —
# the same column twice. Entropy is kept (amount-weighted, guarded at 5
# counterparties); the raw count is carried only as the ratio above.
metrics = [c for c in ["geo_spread_km", "geo_reach_p50_km", "geo_zip3_entropy",
                       "geo_spread_in_km", "geo_spread_out_km"]
           if c in B]
rows = []
for m in metrics:
    r, p, r2, beta = peer_percentile(B, m)
    B[m + "_resid"] = r
    B[m + "_pctile_naics_size"] = p
    rows.append({"metric": m, "R2_size_explains": r2,
                 "beta_log_degree": beta[1], "beta_log_strength": beta[2],
                 "rankable": int(p.notna().sum()),
                 "rankable_pct": p.notna().mean()})
norm = pd.DataFrame(rows)
print(norm.to_string(index=False))
RESULTS["peer_normalisation"] = norm.to_dict("records")
print("\nR2_size_explains is the share of the raw metric that is just size.")
print("v1 measured 0.011-0.060 here: size confounding is REAL BUT SMALL, and "
      "the manifest's premise that raw dispersion is badly size-confounded is "
      "only weakly true. Residualisation still earns its place (it drives the "
      "size correlations to ~0) but it was not the load-bearing step.")

In [ ]:
# Before / after: how much of the size confound did we actually remove?
chk = []
for m in metrics:
    chk.append({"metric": m,
                "raw_vs_log_strength": B[m].corr(np.log1p(B.strength), method="spearman"),
                "resid_vs_log_strength": B[m + "_resid"].corr(np.log1p(B.strength),
                                                              method="spearman"),
                "raw_vs_log_degree": B[m].corr(np.log1p(B.deg_tot), method="spearman"),
                "resid_vs_log_degree": B[m + "_resid"].corr(np.log1p(B.deg_tot),
                                                            method="spearman")})
chk = pd.DataFrame(chk)
print(chk.to_string(index=False))

plot = chk.melt(id_vars="metric", var_name="pair", value_name="spearman")
fig = px.bar(plot, x="metric", y="spearman", color="pair", barmode="group",
             title="Size confounding before and after residualisation "
                   "(Spearman vs log size)", color_discrete_sequence=PALETTE)
fig.add_hline(y=0, line_color="black")
fig.update_layout(height=420, xaxis_title=None)
fig.show()

---
## 5. Does geography say anything the other metrics don't?

The honest test. If `geo_spread_pctile` is just `degree` in disguise, there is
no geographic product here. Spearman on the **peer-normalised** values,
business nodes only, coverage-gated.

In [ ]:
corr_cols = ([m + "_pctile_naics_size" for m in metrics if m + "_pctile_naics_size" in B]
             + [c for c in ["geo_home_zip3_share_in", "geo_home_state_share_in",
                            "geo_home_state_share_out", "geo_R",
                            "geo_registered_vs_flow_km", "geo_cov_amt_in"] if c in B]
             + [c for c in [C["top_share"], C["pagerank"], C["clustering"],
                            C["recip"], C["hub_exp"]] if c and c in B]
             + [c for c in B.columns if c.startswith("share_in_amt_")
                or c.startswith("share_out_amt_")][:6]
             + [c for c in ["naics2_entropy_in", "naics2_entropy_out",
                            "same_naics2_in_share", "same_naics2_out_share"] if c in B]
             + ["strength", "deg_tot"])
corr_cols = [c for c in dict.fromkeys(corr_cols) if c in B]
cm = B[corr_cols].corr(method="spearman", min_periods=500)
short = {c: c.replace("_pctile_naics_size", "·pct").replace("geo_", "")
             .replace("share_", "sh_").replace("_amt", "") for c in corr_cols}
fig = px.imshow(cm.rename(index=short, columns=short), text_auto=".2f",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                title="Spearman correlation — peer-normalised geo vs the rest of "
                      f"the metric set (business, coverage-gated, {REF_MONTH})")
fig.update_layout(height=780)
fig.show()

In [ ]:
# The specific question: what does footprint width relate to, once size is out?
tgt = "geo_spread_km_pctile_naics_size"
if tgt in cm:
    s = cm[tgt].drop(labels=[tgt]).dropna().sort_values()
    top = pd.concat([s.head(8), s.tail(8)]).rename("rho").reset_index()
    top.columns = ["metric", "rho"]
    top["label"] = [short.get(i, i) for i in top.metric]
    fig = px.bar(top, x="rho", y="label", orientation="h", color="rho",
                 color_continuous_scale="RdBu_r", range_color=[-0.6, 0.6],
                 title="Strongest associations with peer-normalised footprint width")
    fig.update_layout(height=520, xaxis_title="Spearman rho", yaxis_title=None,
                      coloraxis_showscale=False)
    fig.show()
    print(s.to_string())
    RESULTS["geo_vs_nongeo_max_abs_rho"] = float(
        s.drop(labels=[c for c in s.index if c.startswith("geo_")
                       or c.endswith("·pct")], errors="ignore").abs().max())
    print(f"\nmax |rho| against any NON-geo metric: "
          f"{RESULTS['geo_vs_nongeo_max_abs_rho']:.3f}")
    print("\nIf |rho| against strength and degree is now near zero while the "
          "composition and homophily terms are not, the peer-normalised "
          "footprint is carrying independent information — which is the "
          "precondition for any of the use cases below being worth building.")

---
## 6. Temporal dynamics — the panel

23 months of footprint per node. This is the part that cannot be done in a
tabular warehouse without the graph, and it is where the usable products are.

**Every business node, no sampling.** Filtering happens on stated analytical
criteria (strength, baseline, coverage) in §7, never on a random subsample —
a sampled denominator makes every peer group approximate for no benefit.

In [ ]:
pcols = [c for c in [C["node"], C["time"], C["node_type"], C["etype"], C["naics2"],
                     C["state"], C["cust_name"], C["in_s"], C["out_s"],
                     C["in_d"], C["out_d"], C["lat"], C["lon"]] if c]
pgeo  = [c for c in ["geo_spread_km", "geo_spread_in_km", "geo_spread_out_km",
                     "geo_centroid_lat", "geo_centroid_lon", "geo_locality_class",
                     "geo_zip3_entropy", "geo_reach_p50_km",
                     "geo_registered_vs_flow_km", "geo_home_state_share_in",
                     "geo_cov_amt_in", "geo_cov_amt_out",
                     "geo_n_cp_located_in", "geo_n_cp_located_out"] if c in GEO]
sel = ", ".join(dict.fromkeys(pcols + pgeo))

sql = f"""
SELECT {sel}
FROM {TABLE}
WHERE {C['version']} = '{VERSION}'
  AND {C['etype']}   = '{BIZ}'
"""
P = q(sql, "temporal panel (all business nodes)", max_rows=12_000_000)
P.columns = [c.lower() for c in P.columns]
P = P.rename(columns={C["node"]: "node", C["time"]: "time_key"})
P["time_key"] = P.time_key.astype(str)
P["strength"] = P[C["in_s"]].fillna(0) + P[C["out_s"]].fillna(0)
P["deg_tot"]  = P[C["in_d"]].fillna(0) + P[C["out_d"]].fillna(0)
P = shrink(P).sort_values(["node", "time_key"])
print(f"panel: {P.node.nunique():,} business nodes x {P.time_key.nunique()} months "
      f"= {len(P):,} rows")

### 6.1 Locality-class transitions — a go/no-go, not a description

The diagonal is stickiness. **If it is not dominant the class is noise**, and
every alert built on it is noise too. This was declared a go/no-go in advance
and the answer is recorded below rather than reinterpreted after the fact.

In [ ]:
if "geo_locality_class" in P:
    t = P[["node", "time_key", "geo_locality_class"]].dropna()
    t["nxt"] = t.groupby("node", observed=True)["geo_locality_class"].shift(-1)
    t = t.dropna(subset=["nxt"])
    order = ["LOCAL", "REGIONAL", "MULTI_MARKET", "NATIONAL"]
    M = (pd.crosstab(t.geo_locality_class, t.nxt, normalize="index")
           .reindex(index=order, columns=order))
    fig = px.imshow(M, text_auto=".1%", color_continuous_scale="Blues",
                    title="Locality-class transition matrix, month over month "
                          "(all business nodes) — diagonal = stickiness")
    fig.update_layout(height=430, xaxis_title="next month", yaxis_title="this month")
    fig.show()
    diag = float(np.nanmean(np.diag(M.to_numpy())))
    RESULTS["locality_diagonal"] = diag
    RESULTS["locality_alerting"] = "DROPPED" if diag < 0.70 else "usable"
    print(f"mean diagonal = {diag:.1%}  ->  class-transition alerting: "
          f"{RESULTS['locality_alerting']}")
    print("\nThe churn is asymmetric toward LOCAL (REGIONAL->LOCAL and "
          "NATIONAL->LOCAL are both large). That is the signature of "
          "month-to-month counterparty SAMPLING, not of firms becoming local: "
          "in a thin month the counterparty cloud collapses and spread falls "
          "toward zero. It is also why §7 tests the continuous series with a "
          "rank-based statistic instead of counting class changes.")

### 6.2 Common-mode drift — is everyone widening at once?

Before attributing a trend to a customer, subtract what the whole panel did.
If the panel median footprint drifts — because coverage improved, or the
graph grew, or a rail changed — every node inherits that slope and the
detector will flag a fixed share of the book every month forever.

Each node's series is expressed **relative to the panel median for that
month**. Both raw and adjusted trends are carried, so the size of the
common-mode component is visible rather than assumed away.

In [ ]:
P["_lsp"] = np.log1p(P["geo_spread_km"])
common = P.groupby("time_key", observed=True)["_lsp"].median()
P["_lsp_adj"] = P["_lsp"] - P["time_key"].map(common)

cm = common.reset_index(); cm.columns = ["time_key", "median_log1p_spread"]
cm["median_spread_km"] = np.expm1(cm.median_log1p_spread)
fig = px.line(cm, x="time_key", y="median_spread_km", markers=True,
              title="Panel-median footprint over time — the common mode every "
                    "node's trend must be measured against",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=360, xaxis_title=None, yaxis_title="median geo_spread_km")
fig.show()
S_cm, z_cm, p_cm, _ = mann_kendall(cm.median_log1p_spread.to_numpy()[None, :],
                                   min_obs=6)
RESULTS["common_mode_p"] = float(p_cm[0])
RESULTS["common_mode_first_last_km"] = [float(cm.median_spread_km.iloc[0]),
                                        float(cm.median_spread_km.iloc[-1])]
print(f"panel median {cm.median_spread_km.iloc[0]:,.1f} km -> "
      f"{cm.median_spread_km.iloc[-1]:,.1f} km | Mann-Kendall p = {p_cm[0]:.4f}")
print("If this is significant, per-node RAW slopes are partly this line, and "
      "the adjusted series is the one to test. Both are computed below.")

### 6.3 How much noise is in a single node's series?

This sets the detector's power, and it is measurable rather than assumed.
The within-node scale is the median absolute month-over-month change in
`log1p(spread)`. Simulating against that noise level gives the smallest
growth rate the detector can actually find — so a low flag count reads
correctly as *"nothing that large happened"* rather than *"the detector is
broken"*.

In [ ]:
d1 = P.groupby("node", observed=True)["_lsp_adj"].diff().abs()
sigma = float(np.nanmedian(d1) * 1.4826)      # MAD -> sd equivalent
RESULTS["within_node_sigma_log"] = sigma
print(f"within-node noise (log units, robust sd): {sigma:.3f}\n")

rng = np.random.default_rng(0)
T_ = P.time_key.nunique(); N_SIM = 20_000
rows = []
for gr in (1.02, 1.05, 1.09, 1.15):
    base = np.exp(rng.normal(np.log(80), sigma, (N_SIM, T_)))
    pl = np.zeros(N_SIM, bool); pl[::10] = True
    Xs = base.copy(); Xs[pl] *= gr ** np.arange(T_)
    Ss, _, ps, _ = mann_kendall(np.log1p(Xs))
    sg, _ = bh_fdr(ps, FDR_Q); up = sg & (Ss > 0)
    rows.append({"growth_per_month": f"{gr-1:.0%}",
                 "total_over_window": f"{gr**T_-1:.0%}",
                 "recall": round((up & pl).sum() / pl.sum(), 3),
                 "precision": round((up & pl).sum() / max(up.sum(), 1), 3)})
power = pd.DataFrame(rows)
print(power.to_string(index=False))
RESULTS["detector_power"] = power.to_dict("records")
print("\nPrecision stays high at every level where anything is detectable at "
      "all — that is BH doing its job. Recall is what the noise buys you, and "
      "it is the honest read on what a low flag count means.")

### 6.4 Centroid drift — moving, as distinct from widening

A firm can widen without moving (new remote customers around the same base)
or move without widening (relocation). Different business events, different
columns; a combined "geo change score" would blur them into one number nobody
can action.

In [ ]:
g = P.groupby("node", sort=False, observed=True)
ncp_tot = (P[[c for c in ("geo_n_cp_located_in", "geo_n_cp_located_out")
              if c in P]].sum(axis=1) if "geo_n_cp_located_in" in P
           else pd.Series(np.nan, index=P.index))
traj = pd.DataFrame({
    "n_months":     g["time_key"].size(),
    "avg_strength": g["strength"].mean(),
    "naics2":       g[C["naics2"]].first() if C["naics2"] else np.nan,
    "state":        g[C["state"]].first() if C["state"] else np.nan,
    "name":         g[C["cust_name"]].first() if C["cust_name"] else "",
    "spread_first": g["geo_spread_km"].first(),
    "spread_last":  g["geo_spread_km"].last(),
    "spread_med":   g["geo_spread_km"].median(),
})
traj["ncp_med"] = ncp_tot.groupby(P["node"], observed=True).median()

if {"geo_centroid_lat", "geo_centroid_lon"} <= set(P.columns):
    P["_plat"] = g["geo_centroid_lat"].shift()
    P["_plon"] = g["geo_centroid_lon"].shift()
    P["drift_km"] = haversine_km(P._plat, P._plon,
                                 P.geo_centroid_lat, P.geo_centroid_lon)
    gd = P.groupby("node", sort=False, observed=True)["drift_km"]
    traj["drift_med_km"]   = gd.median()
    traj["drift_total_km"] = haversine_km(
        g["geo_centroid_lat"].first(), g["geo_centroid_lon"].first(),
        g["geo_centroid_lat"].last(),  g["geo_centroid_lon"].last())
    # Net displacement over cumulative wandering. Bounded by 1 by the triangle
    # inequality — v1 reported a max of 94,115 because the denominator skips
    # NaN, so a node with one non-null hop and distant endpoints blows up.
    # Require enough hops for the ratio to mean anything, then clip.
    n_hops, cum = gd.count(), gd.sum()
    traj["drift_directedness"] = np.where(
        (n_hops >= 3) & (cum > 0),
        np.clip(sdiv(traj.drift_total_km, cum), 0, 1), np.nan)

traj = traj[traj.n_months >= MIN_MONTHS]
print(traj[[c for c in ("n_months", "avg_strength", "ncp_med", "spread_first",
                        "spread_last", "drift_med_km", "drift_total_km",
                        "drift_directedness") if c in traj]]
      .describe().T.to_string())
print("\ndrift_directedness is now bounded by 1 as the geometry requires "
      "(v1 max was 94,115 — an unguarded ratio against a NaN-skipping sum).")

---
## 7. USE CASE A — Footprint Expansion / Contraction Monitor
### *"Which clients' geographic footprint changed materially this quarter?"*

> **v1 measured 70.6% false discovery against its own placebo — and that is
> not a threshold set too loose.** The v1 rule required the peer-relative
> slope percentile to exceed 0.90, which flags 10% of the population *by
> construction*, signal or no signal, combined with a level-shift test a coin
> flip passes 12.5% of the time. **A percentile cutoff cannot control false
> discovery at any setting**, because it defines the flagged fraction rather
> than measuring it.

**v2 replaces the rule with a test that has a null distribution.**

| step | what it does |
|---|---|
| **Mann-Kendall** on panel-adjusted `log1p(geo_spread_km)` | rank-based trend statistic with an *analytic* null; immune to the one-month spike a single large remote payment produces, and tie-corrected because a co-located cloud gives spread exactly 0 and those zeros tie heavily |
| **Benjamini-Hochberg** at `FDR_Q` | of the nodes flagged, at most `FDR_Q` are expected false — whatever the population size. This is the number to quote to a stakeholder |
| **Effect size** — trailing/leading ratio ≥ `MIN_EFFECT_RATIO` | significance ≠ materiality. A firm can widen detectably by 3% and nobody cares |
| **Coverage-trend exclusion** | a node whose *located-dollar coverage* is itself trending will appear to widen. Tested with the same statistic and excluded when it trends the same way — the most likely false-positive source, and invisible without the test |
| **Eligibility floors** — strength, sustained counterparties, months | applied *before* testing. Independent of whether a series trends, so they do not bias the p-values; they cut multiplicity and raise power |
| **Baseline split** | `lead_km < MIN_SPREAD_BASELINE` is **market entry**, not widening. v1's top-15 was dominated by nodes starting at 0.0 km, where a first out-of-market counterparty produces an enormous slope off a near-zero base |

In [ ]:
# ---- eligibility, applied BEFORE testing (independent of trend) ----------
W = P[P.node.isin(traj.index)].copy()
W["cov_max"] = W[[c for c in ("geo_cov_amt_in", "geo_cov_amt_out")
                  if c in W]].max(axis=1)
gw = W.groupby("node", observed=True)
elig = traj.join(pd.DataFrame({"cov_min": gw["cov_max"].min(),
                               "cov_med": gw["cov_max"].median()}))
steps = [("panel", len(elig))]
m = elig.cov_min.ge(MIN_GEO_COV).fillna(False);      steps.append(("coverage every month", int(m.sum())))
m &= elig.ncp_med.ge(MIN_CP_SUSTAINED).fillna(False); steps.append((f"cp>={MIN_CP_SUSTAINED} sustained", int(m.sum())))
m &= elig.avg_strength.ge(MIN_STRENGTH_PANEL);       steps.append((f"strength>={MIN_STRENGTH_PANEL:,}", int(m.sum())))
m &= elig.spread_med.notna();                        steps.append(("spread measurable", int(m.sum())))
ELIG = elig[m]
print(" -> ".join(f"{k}: {v:,}" for k, v in steps))
RESULTS["eligibility_funnel"] = steps
print(f"\ntesting {len(ELIG):,} nodes. Every floor here is independent of "
      f"whether the series trends, so the p-values stay valid while the "
      f"multiplicity burden drops from {len(elig):,} tests to {len(ELIG):,}.")

In [ ]:
# ---- pivot to (node x month) matrices and test --------------------------
months = sorted(P.time_key.unique())
WE = W[W.node.isin(ELIG.index)]
def to_matrix(col):
    return (WE.pivot_table(index="node", columns="time_key", values=col,
                           aggfunc="mean", observed=True)
              .reindex(columns=months))

Msp, Mraw, Mcov = to_matrix("_lsp_adj"), to_matrix("_lsp"), to_matrix("cov_max")
nodes = Msp.index

S_sp, z_sp, p_sp, n_sp = mann_kendall(Msp.to_numpy())
S_raw, _, p_raw, _     = mann_kendall(Mraw.to_numpy())
S_cov, _, p_cov, _     = mann_kendall(Mcov.to_numpy())
slope = theilsen_slope_matrix(Msp.to_numpy())

k = max(3, MIN_MONTHS // 4)
R = np.expm1(Mraw.to_numpy())
lead  = np.nanmedian(R[:, :k], axis=1)
trail = np.nanmedian(R[:, -k:], axis=1)

T7 = pd.DataFrame({
    "mk_S": S_sp, "mk_z": z_sp, "mk_p": p_sp, "n_obs": n_sp,
    "mk_p_raw": p_raw, "slope_log_per_month": slope,
    "lead_km": lead, "trail_km": trail,
    "effect_ratio": sdiv(trail, np.maximum(lead, 1e-9)),
    "cov_S": S_cov, "cov_p": p_cov,
}, index=nodes).join(ELIG)

sig, crit = bh_fdr(T7.mk_p.to_numpy(), FDR_Q)
T7["bh_significant"] = sig
T7["cov_trend_conflict"] = (T7.cov_p < 0.05) & (np.sign(T7.cov_S) == np.sign(T7.mk_S))
RESULTS["bh_critical_p"] = crit
print(f"BH critical p = {crit:.3e} at q={FDR_Q}  "
      f"-> {int(sig.sum()):,} of {len(T7):,} nodes significant")
print(f"excluded for a coverage trend in the same direction: "
      f"{int(T7.cov_trend_conflict.sum()):,}")
print(f"raw vs panel-adjusted disagreement on significance: "
      f"{int(((T7.mk_p < crit) != (T7.mk_p_raw < crit)).sum()):,} nodes "
      f"— the size of the common-mode effect on the flag list")

In [ ]:
# ---- three queues, deliberately separate --------------------------------
base_ok = T7.lead_km >= MIN_SPREAD_BASELINE
core    = T7.bh_significant & ~T7.cov_trend_conflict

EXPAND   = T7[core & (T7.mk_S > 0) & base_ok
              & (T7.effect_ratio >= MIN_EFFECT_RATIO)].copy()
CONTRACT = T7[core & (T7.mk_S < 0) & base_ok
              & (T7.effect_ratio <= 1 / MIN_EFFECT_RATIO)].copy()
ENTRY    = T7[core & (T7.mk_S > 0) & ~base_ok
              & (T7.trail_km >= MIN_SPREAD_BASELINE)].copy()

for nm, df in (("EXPANDING", EXPAND), ("CONTRACTING", CONTRACT),
               ("MARKET ENTRY", ENTRY)):
    print(f"{nm:<14}: {len(df):>7,} nodes ({len(df)/max(len(T7),1):.2%} of tested)")
RESULTS["queues"] = {"tested": int(len(T7)), "expanding": int(len(EXPAND)),
                     "contracting": int(len(CONTRACT)), "entry": int(len(ENTRY))}
print(f"\nExpected false positives among the significant set: <= {FDR_Q:.0%}, "
      f"by construction rather than by hope.")
cols = ["name", "naics2", "state", "avg_strength", "lead_km", "trail_km",
        "effect_ratio", "slope_log_per_month", "mk_p", "n_obs", "drift_total_km"]
EXPAND.sort_values("effect_ratio", ascending=False).head(15)[
    [c for c in cols if c in EXPAND]]

In [ ]:
# Trajectories of the largest expanders against the panel median.
if len(EXPAND):
    pick = EXPAND.sort_values("avg_strength", ascending=False).head(8).index
    tr = P[P.node.isin(pick)][["node", "time_key", "geo_spread_km", C["cust_name"]]]
    fig = px.line(tr, x="time_key", y="geo_spread_km", color="node", markers=True,
                  hover_data=[C["cust_name"]], color_discrete_sequence=PALETTE,
                  title="Footprint expansion candidates vs the panel median "
                        "(BH-significant, material effect, baseline-qualified)")
    fig.add_scatter(x=cm.time_key, y=cm.median_spread_km, name="panel median",
                    line=dict(color="black", dash="dash", width=3))
    fig.update_layout(height=460, yaxis_title="geo_spread_km", xaxis_title=None)
    fig.show()

### 7.1 Calibrating the detector — placebo test

Permuting each node's series in time destroys any trend while preserving that
node's distribution exactly, so **every flag raised on permuted data is false
by construction**. v1 scored 70.6% here. Under BH the expected answer is at
or below `FDR_Q`, and the placebo now *verifies* the guarantee instead of
being used to hunt for a threshold.

In [ ]:
Araw = Msp.to_numpy()
real_rate = float(T7.bh_significant.mean())
perm = []
for seed in range(3):
    r = np.random.default_rng(seed)
    Ap = np.apply_along_axis(r.permutation, 1, Araw)
    _, _, pp, _ = mann_kendall(Ap)
    sp, _ = bh_fdr(pp, FDR_Q)
    perm.append(float(sp.mean()))
perm_rate = float(np.mean(perm))
fdr_est = perm_rate / real_rate if real_rate > 0 else np.nan
RESULTS["placebo"] = {"real_flag_rate": real_rate,
                      "permuted_flag_rate": perm_rate,
                      "estimated_fdr": fdr_est}
print(f"flag rate on REAL panel     : {real_rate:.3%}")
print(f"flag rate on PERMUTED panel : {perm_rate:.3%}  (all false by construction)")
print(f"estimated false-discovery   : {fdr_est:.1%} of flags    [v1: 70.6%]")
print(f"\nBH guarantees <= {FDR_Q:.0%}. If the measured value sits far below "
      f"it, the test is conservative and FDR_Q can be raised to recover "
      f"recall — a decision about how much analyst time a false lead costs, "
      f"which belongs to whoever works the queue.")

**App shape.** A monthly-refreshed queue, not a dashboard. Columns: customer,
sector, lead→trail km, effect ratio, MK p-value, months tested, sparkline.
Three separate lists — expanding, contracting, market entry — because they go
to different people. One row per customer per event, closed when the trend
breaks; a list that re-alerts the same customer every month is ignored by
week three.

---
## 8. USE CASE B — Registered-vs-Flow Audit
### *"Is this customer's registered address where its business actually is?"*

> **v1 returned 52 nodes (0.28%) — not a queue.** The cause is visible in the
> §5 correlation matrix: `registered_vs_flow_km` and footprint width
> correlate at **0.76**. A large gap almost always accompanies a wide
> footprint, so requiring *gap above p90* **and** *spread below the median*
> intersects a nearly empty corner of a strongly diagonal cloud.

The question was never "is the gap large" — it is **"is the gap larger than
this customer's own footprint width predicts."** That is a residual, not an
AND of two marginal thresholds.

In [ ]:
gapc = "geo_registered_vs_flow_km"
if gapc in G and "geo_spread_km" in G:
    A = G.dropna(subset=[gapc, "geo_spread_km"]).copy()
    A = A[(A[gapc] > 0) & (A.geo_spread_km > 0)]
    x = np.log(A.geo_spread_km.to_numpy(dtype="float64"))
    y = np.log(A[gapc].to_numpy(dtype="float64"))
    b1, b0 = np.polyfit(x, y, 1)
    A["gap_resid"] = y - (b0 + b1 * x)
    A["gap_resid_z"] = A.gap_resid / A.gap_resid.std()
    A["mislocated"] = A.gap_resid >= A.gap_resid.quantile(0.99)
    MIS = A[A.mislocated]
    RESULTS["mislocated"] = {"n": int(len(MIS)),
                             "share": float(A.mislocated.mean()),
                             "beta_log_spread": float(b1),
                             "r2": float(np.corrcoef(x, y)[0, 1] ** 2)}
    print(f"log(gap) = {b0:.2f} + {b1:.2f}*log(spread)   R2 = "
          f"{RESULTS['mislocated']['r2']:.3f}")
    print(f"MISLOCATED (top 1% of residual): {len(MIS):,} business nodes "
          f"({A.mislocated.mean():.2%})  [v1's AND-of-marginals returned 52]")
    s = A.sample(60000, random_state=0) if len(A) > 60000 else A
    fig = px.scatter(s, x=s.geo_spread_km.clip(1, 5000),
                     y=s[gapc].clip(1, 5000), color="gap_resid_z",
                     opacity=0.35, log_x=True, log_y=True,
                     color_continuous_scale="RdBu_r", range_color=[-3, 3],
                     labels={"x": "geo_spread_km (footprint width)",
                             "y": "registered vs flow gap (km)",
                             "color": "residual z"},
                     title="Registered address representativeness — the flag is "
                           "distance ABOVE the fitted line, not raw gap")
    xs = np.linspace(np.log(1), np.log(5000), 50)
    fig.add_scatter(x=np.exp(xs), y=np.exp(b0 + b1 * xs), mode="lines",
                    name="expected gap", line=dict(color="black", width=2))
    fig.update_layout(height=560)
    fig.show()
    print("\nNote the scatter is a sample for rendering only — the fit, the "
          "residuals and the queue are computed on the full population.")

In [ ]:
if "MIS" in dir() and {"geo_centroid_lat", "geo_centroid_lon"} <= set(G.columns):
    top = MIS.nlargest(300, "strength")
    fig = go.Figure()
    for _, r in top.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[r[C["lon"]], r.geo_centroid_lon],
            lat=[r[C["lat"]], r.geo_centroid_lat], mode="lines",
            line=dict(width=1, color="rgba(200,60,60,.35)"),
            showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scattergeo(lon=top[C["lon"]], lat=top[C["lat"]],
        mode="markers", marker=dict(size=5, color="#1f77b4"),
        name="registered address",
        text=top[C["cust_name"]] if C["cust_name"] else None))
    fig.add_trace(go.Scattergeo(lon=top.geo_centroid_lon, lat=top.geo_centroid_lat,
        mode="markers", marker=dict(size=5, color="#d62728", symbol="diamond"),
        name="flow centroid",
        text=top[C["cust_name"]] if C["cust_name"] else None))
    fig.update_layout(height=620, geo=dict(scope="usa", landcolor="#f5f5f5"),
                      title="Top 300 mislocated business customers by strength — "
                            "registered pin vs where the money actually is")
    fig.show()
    print("Next step for production: reverse-geocode the flow centroid to a "
          "CBSA and emit the registered-CBSA -> flow-CBSA reassignment matrix. "
          "That matrix IS the territory audit deliverable; this map is only "
          "the origin side of it.")

---
## 9. USE CASE C — Trade-Role Taxonomy from footprint asymmetry

> **v1 cut the roles at the median, which is ~10 km.** Calling a customer
> whose suppliers are 11 km away "national" is indefensible, and a median cut
> forces an even 30/30/20/20 split by construction. The locality classes
> already define the thresholds the rest of the project uses; v2 uses those.

`geo_spread_in_km` and `geo_spread_out_km` are the **revenue footprint** and
the **supply footprint**. Their asymmetry is a behavioural role independent
of NAICS — usable both as a segmentation and as a cross-check on the label.

In [ ]:
LOCALITY_CUT = 250.0     # km — the REGIONAL / MULTI_MARKET boundary
if {"geo_spread_in_km", "geo_spread_out_km"} <= set(G.columns):
    T = G.dropna(subset=["geo_spread_in_km", "geo_spread_out_km"]).copy()
    wide_in  = T.geo_spread_in_km  > LOCALITY_CUT
    wide_out = T.geo_spread_out_km > LOCALITY_CUT
    T["role"] = np.select(
        [~wide_in & ~wide_out, ~wide_in & wide_out, wide_in & ~wide_out],
        ["community", "local_revenue_distant_supply",
         "national_revenue_local_cost"], "national_intermediary")
    vc = T.role.value_counts(normalize=True)
    RESULTS["trade_roles"] = vc.to_dict()
    print(vc.to_string())
    print(f"\nCut at {LOCALITY_CUT:.0f} km (the REGIONAL/MULTI_MARKET boundary), "
          f"not the median. v1's median cut was ~10 km, which forced an even "
          f"split and made every label meaningless.")
    s = T.sample(60000, random_state=0) if len(T) > 60000 else T
    fig = px.scatter(s, x=s.geo_spread_in_km.clip(1, 5000),
                     y=s.geo_spread_out_km.clip(1, 5000), color="role",
                     opacity=0.3, log_x=True, log_y=True,
                     color_discrete_sequence=PALETTE,
                     labels={"x": "revenue footprint — geo_spread_in_km",
                             "y": "supply footprint — geo_spread_out_km"},
                     title="Trade-role taxonomy from footprint asymmetry "
                           "(business, coverage-gated, absolute km cuts)")
    fig.add_vline(x=LOCALITY_CUT, line_dash="dash", line_color="grey")
    fig.add_hline(y=LOCALITY_CUT, line_dash="dash", line_color="grey")
    fig.update_layout(height=580)
    fig.show()

In [ ]:
if "T" in dir() and "role" in T and C["naics2"]:
    topn = T[C["naics2"]].value_counts().head(15).index
    ct = pd.crosstab(T.loc[T[C['naics2']].isin(topn), C["naics2"]],
                     T.loc[T[C['naics2']].isin(topn), "role"], normalize="index")
    fig = px.imshow(ct, text_auto=".0%", aspect="auto",
                    color_continuous_scale="Oranges",
                    title="Trade role by NAICS2 — cells far from their sector's "
                          "row profile are the mislabelled-or-interesting cases")
    fig.update_layout(height=520, xaxis_title=None, yaxis_title="naics2")
    fig.show()
    rate = ct.stack().rename("sector_role_rate").reset_index()
    rate.columns = [C["naics2"], "role", "sector_role_rate"]
    T2 = T.merge(rate, on=[C["naics2"], "role"], how="left")
    ODD = T2[T2.sector_role_rate < 0.05]
    RESULTS["role_sector_outliers"] = int(len(ODD))
    print(f"\n{len(ODD):,} customers whose trade role occurs in <5% of their "
          f"own sector — mislabelled NAICS, or a genuinely unusual model. "
          f"Both are worth a call:")
    print(ODD.nlargest(15, "strength")[
        [c for c in (C["cust_name"], C["naics2"], "role", "geo_spread_in_km",
                     "geo_spread_out_km", "strength") if c]].to_string(index=False))

---
## 10. USE CASE D — Distance-weighted counterparty concentration

Concentration and distance are monitored separately today. Together they are
a sharper statement: a customer whose revenue is concentrated in a few
counterparties who are all far away has neither diversification nor
proximity. Kept as a 2-D view rather than a blended index, so it stays
visible which of the two drove the flag.

In [ ]:
conc = C["top_share"]
if conc and conc in G and "geo_reach_p50_km" in G:
    Rk = G.dropna(subset=[conc, "geo_reach_p50_km"]).copy()
    Rk["conc_pct"]  = Rk[conc].rank(pct=True)
    Rk["reach_pct"] = Rk.geo_reach_p50_km.rank(pct=True)
    Rk["flag"] = (Rk.conc_pct >= 0.90) & (Rk.reach_pct >= 0.90)
    RESULTS["conc_distance_flagged"] = int(Rk.flag.sum())
    s = Rk.sample(60000, random_state=0) if len(Rk) > 60000 else Rk
    fig = px.density_heatmap(s, x="reach_pct", y="conc_pct", nbinsx=40, nbinsy=40,
                             color_continuous_scale="Magma",
                             labels={"reach_pct": "distance percentile (p50 reach)",
                                     "conc_pct": "concentration percentile"},
                             title="Concentration x distance — the top-right cell "
                                   "is the queue")
    fig.add_hline(y=0.90, line_dash="dash", line_color="white")
    fig.add_vline(x=0.90, line_dash="dash", line_color="white")
    fig.update_layout(height=520); fig.show()
    print(f"flagged: {int(Rk.flag.sum()):,} nodes ({Rk.flag.mean():.2%})")
    print(Rk[Rk.flag].nlargest(12, "strength")[
        [c for c in (C["cust_name"], C["naics2"], C["state"], conc,
                     "geo_reach_p50_km", "geo_spread_km", "strength") if c]
    ].to_string(index=False))
else:
    print("concentration column not resolved — skipping §10.")

---
## 11. Rung agreement — P99_9 vs P99

Standing rule: no result is reportable until computed at two adjacent rungs
and shown to agree. Parameterised here; the full three-rung comparison is a
separate run.

In [ ]:
sql = f"""
SELECT {C['version']} AS version, {C['node_type']} AS node_type,
       COUNT(*) AS n_nodes,
       AVG(geo_spread_km) AS mean_spread,
       PERCENTILE_APPROX(geo_spread_km, 0.5) AS med_spread,
       PERCENTILE_APPROX(geo_reach_p50_km, 0.5) AS med_reach_p50,
       AVG(geo_home_state_share_in) AS mean_home_state_in
FROM {TABLE}
WHERE {C['version']} IN ('{VERSION}', '{VERSION_ALT}')
  AND {C['time']} = '{REF_MONTH}'
GROUP BY 1, 2
"""
rung = q(sql, "rung agreement")
cmp = rung.pivot(index="node_type", columns="version", values="med_spread")
if {VERSION, VERSION_ALT} <= set(cmp.columns):
    cmp["ratio"] = sdiv(cmp[VERSION_ALT], cmp[VERSION])
    print(cmp.to_string())
    RESULTS["rung_median_spread_ratio"] = {k: float(v) for k, v in
                                           cmp["ratio"].dropna().items()}
print("\nRatios near 1.0 = the finding survives de-hubbing. Expect the §7 "
      "queues to be LESS rung-stable than the cross-sectional metrics: hub "
      "removal changes which counterparties define the cloud each month, and "
      "therefore changes the trend itself, not just its level.")

---
## 12. Results — written to disk

Queues as parquet for downstream use, plus a dated `RESULTS.md` and a run
manifest carrying the config and every headline number, so any figure can be
traced back to the run that produced it.

In [ ]:
import json, datetime
os.makedirs(OUT, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
written = []

def dump(df, name, cols=None, index=True):
    if df is None or not len(df):
        print(f"  [skip] {name} — empty"); return
    d = df[[c for c in (cols or df.columns) if c in df]].copy()
    d.to_parquet(os.path.join(OUT, name), index=index)
    written.append((name, len(d)))
    print(f"  {name:<40} {len(d):>9,} rows")

print(f"writing to {OUT}")
qcols = ["name", "naics2", "state", "avg_strength", "lead_km", "trail_km",
         "effect_ratio", "slope_log_per_month", "mk_p", "mk_z", "n_obs",
         "cov_med", "ncp_med", "drift_total_km", "drift_directedness"]
dump(EXPAND,   f"expansion_queue_{VERSION}.parquet",    qcols)
dump(CONTRACT, f"contraction_queue_{VERSION}.parquet",  qcols)
dump(ENTRY,    f"market_entry_queue_{VERSION}.parquet", qcols)
dump(T7,       f"trend_tests_{VERSION}.parquet")
if "MIS" in dir():
    dump(MIS, f"mislocated_queue_{VERSION}_{REF_MONTH}.parquet",
         [C["node"], C["cust_name"], C["naics2"], C["state"], "geo_spread_km",
          gapc, "gap_resid", "gap_resid_z", "strength"], index=False)
if "T" in dir() and "role" in T:
    dump(T, f"trade_roles_{VERSION}_{REF_MONTH}.parquet",
         [C["node"], C["cust_name"], C["naics2"], "role", "geo_spread_in_km",
          "geo_spread_out_km", "strength"], index=False)
pctc = [c for c in B.columns if c.endswith("_pctile_naics_size")]
dump(B, f"peer_percentiles_{VERSION}_{REF_MONTH}.parquet",
     [C["node"], C["naics2"], "size_decile"] + pctc, index=False)
comp.to_csv(os.path.join(OUT, f"composition_by_month_{VERSION}.csv"), index=False)
written.append((f"composition_by_month_{VERSION}.csv", len(comp)))
print(f"  composition_by_month_{VERSION}.csv{'':<8} {len(comp):>9,} rows")

In [ ]:
ind = comp[(comp.time_key.astype(str) == REF_MONTH) &
           (comp.node_type == "individual")]
RESULTS.update({"version": VERSION, "ref_month": REF_MONTH, "generated": stamp,
    "individual_share_nodes": float(ind.node_share.iloc[0]) if len(ind) else None,
    "individual_share_dollars": float(ind.dollar_share.iloc[0]) if len(ind) else None})

def pc(x, d=1):
    return "n/a" if x is None or not np.isfinite(x) else f"{x:.{d}%}"

L = [
 "# PKG Geographic Analytics — Results",
 f"*{VERSION} · reference month {REF_MONTH} · generated {stamp}*",
 "",
 "> **Scope: `on_us_c2c`.** Every figure is a position versus other PNC "
 "customers, not versus the economy. `strength = in + out` double-counts and "
 "is a weight, never a volume figure. Single-rung — re-run at V0 and P99 "
 "before briefing.",
 "",
 "## Population",
 "",
 f"- Individual share: **{pc(RESULTS['individual_share_nodes'])} of nodes**, "
 f"{pc(RESULTS['individual_share_dollars'])} of dollars. The pre-`party_type` "
 f"figures were 94.26% / 48.4%; the correction is one-directional "
 f"(person-named organisations move individual → business) and **this "
 f"replaces them**.",
 f"- Locality-class stickiness: **{pc(RESULTS.get('locality_diagonal'))}** mean "
 f"diagonal → class-transition alerting **{RESULTS.get('locality_alerting','?')}**.",
 f"- Within-node noise: **{RESULTS.get('within_node_sigma_log', float('nan')):.3f}** "
 f"log units. Detector power at that noise level:",
 "",
 "| growth/month | over window | recall | precision |",
 "|---|---|---|---|",
] + [f"| {r['growth_per_month']} | {r['total_over_window']} | {r['recall']:.0%} "
     f"| {r['precision']:.0%} |" for r in RESULTS.get("detector_power", [])] + [
 "",
 "## Detector (§7)",
 "",
 f"- Eligibility funnel: " + " → ".join(f"{k} {v:,}" for k, v in
                                        RESULTS.get("eligibility_funnel", [])),
 f"- BH critical p at q={FDR_Q}: **{RESULTS.get('bh_critical_p', float('nan')):.3e}**",
 f"- **Expanding {RESULTS['queues']['expanding']:,}** · "
 f"contracting {RESULTS['queues']['contracting']:,} · "
 f"market entry {RESULTS['queues']['entry']:,} "
 f"(of {RESULTS['queues']['tested']:,} tested)",
 f"- Placebo false-discovery: **{pc(RESULTS['placebo']['estimated_fdr'])}** "
 f"— v1's peer-percentile rule measured **70.6%**. A percentile cutoff flags "
 f"a fixed fraction of the population whether or not anything happened; BH "
 f"bounds the false fraction at {FDR_Q:.0%} by construction.",
 "",
 "## Use cases",
 "",
 f"- **Mislocated** (residual of log gap on log spread, top 1%): "
 f"**{RESULTS.get('mislocated',{}).get('n',0):,}** nodes. Gap–spread "
 f"elasticity {RESULTS.get('mislocated',{}).get('beta_log_spread',float('nan')):.2f}, "
 f"R² {RESULTS.get('mislocated',{}).get('r2',float('nan')):.3f}. v1's "
 f"AND-of-marginals returned 52 because gap and spread correlate at 0.76.",
 f"- **Trade roles** (absolute {int(LOCALITY_CUT)} km cuts): " +
 ", ".join(f"{k} {v:.1%}" for k, v in RESULTS.get("trade_roles", {}).items()),
 f"- Role-vs-sector outliers (<5% of own sector): "
 f"**{RESULTS.get('role_sector_outliers',0):,}**",
 f"- Concentration × distance flagged: **{RESULTS.get('conc_distance_flagged',0):,}**",
 "",
 "## Known limits",
 "",
 "- Geographic metrics use **located counterparties only**; the analysable "
 "population is much smaller than the node count (spread needs ≥2 "
 "counterparties, entropy ≥5).",
 "- Off-us flow is invisible. The expansion monitor will systematically miss "
 "expansion into markets where PNC has no deposit presence — which is where "
 "expansion is most likely. Revisit when PAYS_CPTY lands.",
 "- A moving centroid can be the customer relocating, its counterparties "
 "relocating, or one large new relationship. Separating those needs "
 "edge-level attribution.",
 "",
 "## Files",
 "",
] + [f"- `{n}` ({r:,} rows)" for n, r in written]

path_md = os.path.join(OUT, f"RESULTS_{VERSION}_{REF_MONTH}.md")
with open(path_md, "w") as f:
    f.write("\n".join(L))
with open(os.path.join(OUT, f"run_manifest_{VERSION}.json"), "w") as f:
    json.dump({"generated": stamp, "version": VERSION, "ref_month": REF_MONTH,
               "config": {k: globals()[k] for k in
                          ("FDR_Q", "MIN_EFFECT_RATIO", "MIN_SPREAD_BASELINE",
                           "MIN_STRENGTH_PANEL", "MIN_CP_SUSTAINED",
                           "MIN_GEO_COV", "MIN_CP", "MIN_MONTHS",
                           "LOCALITY_CUT", "PARTY_TYPE_WINS"
                           ) if k in globals()},
               "results": RESULTS}, f, indent=2, default=str)
print("\n".join(L))
print(f"\n-> {path_md}")
print(f"-> {OUT}/run_manifest_{VERSION}.json")

---
## 13. What is worth building

| # | Product | Serves | Effort | Verdict |
|---|---|---|---|---|
| **B** | **Registered-vs-Flow audit** | TM Sales ops, Servicing | **Low** — one column plus a regression | **Build first.** Now that the flag is a residual rather than an AND of marginals it returns a workable population. Territory misassignment is a concrete, checkable error with an owner |
| **A** | **Footprint Expansion / Contraction Monitor** | TM Sales, Risk | Medium | **Build second, on the BH output only.** The v1 rule was 70.6% noise; the FDR guarantee is what makes this shippable at all. Ship the three queues separately — they go to different people |
| **C** | **Trade-role taxonomy** | Product, TM Sales | Medium | A **segmentation attribute**, not a screen. Its value is joining to product holdings to find who has the wrong stack, which needs CRM data this notebook does not have |
| **D** | **Distance-weighted concentration** | Risk | Low | Two columns folded into an existing risk view. Not worth its own surface |

### Sequencing

1. **Close §4 into the pipeline.** Peer percentiles are the reportable
   quantity for three of the four use cases and are currently computed here.
   They belong in `pkg_geo_metrics.py` as `{metric}_pctile_naics_size` with
   the peer-group size recorded alongside — otherwise every app re-derives
   them and they drift apart.
2. **Ship B as a monthly table first, not an app.** One reviewer, one month
   of the mislocated queue, will establish whether the flag is right far
   faster than a Streamlit page will.
3. **A needs a CBSA join** to turn "widened from 40 km to 300 km" into
   "entered the Columbus market" — the form Sales can act on. That is the FI
   Pinning Registry spine; build it once and both A and B consume it.
4. **Then the panel**: customer search → footprint map (registered pin, flow
   centroid, counterparty cloud), 23-month sparklines, peer percentile bars,
   alert history.

### Method notes worth carrying to other modules

- **A percentile cutoff is not a detector.** It flags a fixed fraction of the
  population by construction. Anywhere the project ranks customers and takes
  a top slice — rail shift, behavioural drift, subrogation candidates — the
  same 70.6% failure is available, and the same fix applies: a statistic with
  a null, then BH.
- **Test the confounder with the same statistic as the signal.** The
  coverage-trend exclusion in §7 is cheap and removes the most plausible
  false-positive mechanism. The generalisation: whenever a metric has a
  denominator that can itself drift, trend-test the denominator too.
- **Permutation placebos are close to free** and turn "we flagged N" into "we
  flagged N, of which ≤ q are expected false". Worth adding to every
  candidate-generation module in the project.

### Governance

A queue that routes Sales attention by geography will be asked whether it has
disparate impact. Get the fair-lending / CRA read **before** the surface
exists, and keep `entity_type = 'business'` on all of it — both the
analytically correct filter and the one that keeps consumer-protection
exposure out of scope.